# Related-Work-Adaption: gbert-large **BERT-CRF** NER + HPO

Dieses Notebook baut auf dem *Related-Work*-Modell (spaCy-NER auf `deepset/gbert-large`,
Datensatz **GPTNERMED**, Entitaeten *Medikation / Dosis / Diagnose*) auf und fuehrt zwei
Adaptionen ein:

1. **CRF-Kopf statt reinem Softmax.**
   Die spaCy-Pipeline nutzt einen transition-based Parser; hier wird stattdessen ein
   klassischer Token-Classifier gebaut: `gbert-large`-Encoder -> Linear (Emissionen) ->
   **Conditional-Random-Field (CRF)**. Der CRF ersetzt die *tokenweise unabhaengige*
   Softmax-/Cross-Entropy-Entscheidung durch ein Sequenzmodell mit gelernten
   Label-Uebergaengen. Das erzwingt konsistente BIO-Sequenzen (z. B. kein `I-Dosis`
   ohne vorheriges `B-/I-Dosis`) und verbessert die Entitaets-Grenzen.

2. **Sinnvolles HPO-Buendel (Optuna).**
   Bayesianische Suche (TPE-Sampler + MedianPruner) ueber Lernrate, Weight-Decay,
   Warmup, Dropout, Batch-Size, Epochen und eine *separate CRF-Lernrate*. Danach
   finales Training mit den besten Parametern und Test-Evaluierung (seqeval).

> **Colab:** Runtime -> GPU (T4 reicht). Volles gbert-large ist gross; die HPO laeuft
> daher auf einem Train-Subset mit wenigen Trials, das finale Training auf dem
> kompletten Datensatz.


In [ ]:
!nvidia-smi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
REPO = "https://github.com/wieland-github/information_extraction_german_medical_data.git"
!rm -rf /content/information_extraction_german_medical_data
!git clone {REPO} /content/information_extraction_german_medical_data
%cd /content/information_extraction_german_medical_data

## Abhaengigkeiten

`datasets < 3.0` ist noetig, weil `jfrei/GPTNERMED` ueber ein Loading-Script geladen
wird (`trust_remote_code`). Der CRF kommt aus `pytorch-crf` (`import torchcrf`).

In [ ]:
!pip install -q \
  "transformers>=4.40,<4.46" \
  "datasets>=2.16.0,<3.0.0" \
  "accelerate>=0.30" \
  "seqeval" \
  "optuna>=3.5" \
  "pytorch-crf"
# Sanity-Check der Kern-Imports
import transformers, datasets, torchcrf, optuna, seqeval
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("optuna      ", optuna.__version__)

## Konfiguration

Zentrale Schalter. `TEST=True` fuehrt einen schnellen Smoke-Test aus (winziges Subset,
1 Epoche, 2 Trials), um die gesamte Pipeline in wenigen Minuten zu pruefen.

In [ ]:
import torch, os

MODEL_NAME  = "deepset/gbert-large"     # gleiches Backbone wie das Related-Work-Modell
DATASET     = "jfrei/GPTNERMED"
MAX_LENGTH  = 256
SEED        = 42

TEST = False   # True = Schnelltest (kleines Subset, 1 Epoche, 2 Trials)

# HPO-Budget
N_TRIALS        = 2   if TEST else 8      # Anzahl Optuna-Trials
HPO_TRAIN_LIMIT = 200 if TEST else 3000   # Trainingsbeispiele je Trial (Subset)
HPO_EPOCHS      = 1   if TEST else 3      # Epochen je Trial

# Finales Training
FINAL_EPOCHS = 1 if TEST else 5
FINAL_SEEDS  = [42] if TEST else [42]     # optional mehrere Seeds, z. B. [42, 52, 62]

DRIVE_DIR = "/content/drive/MyDrive/gbert_colab_crf"
os.makedirs(DRIVE_DIR, exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
FP16   = torch.cuda.is_available()
print("Device:", DEVICE, "| fp16:", FP16, "| TEST:", TEST)

## Label-Schema (BIO)

GPTNERMED liefert Zeichen-Spans mit Klassen-IDs (`0=Medikation, 1=Dosis, 2=Diagnose`).
Daraus bauen wir ein BIO-Tagset. `O` erhaelt bewusst die ID `0` (der CRF nutzt `O` auch
als Ersatz-Label fuer maskierte Positionen).

In [ ]:
ID2CLASS = {0: "Medikation", 1: "Dosis", 2: "Diagnose"}

LABEL_LIST = ["O"]
for _cid in sorted(ID2CLASS):
    LABEL_LIST += [f"B-{ID2CLASS[_cid]}", f"I-{ID2CLASS[_cid]}"]

LABEL2ID = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL = {i: l for l, i in LABEL2ID.items()}
NUM_LABELS = len(LABEL_LIST)
assert LABEL2ID["O"] == 0
print(NUM_LABELS, "Labels:", LABEL_LIST)

## Daten laden & Char-Spans -> BIO-Subword-Labels

Wir tokenisieren mit dem gbert-Tokenizer und richten die Zeichen-Spans ueber das
`offset_mapping` auf Subword-Tokens aus:

- Spezial-Tokens (`[CLS]`, `[SEP]`) und Padding bekommen `-100` (aus der Metrik/dem
  CRF-Verlust ausgeschlossen bzw. maskiert).
- Das **erste** Subword einer Entitaet erhaelt `B-…`, weitere Subwords derselben
  Entitaet `I-…`.
- Ein Token gehoert zu einer Entitaet, wenn es sich mit deren Zeichen-Span ueberlappt
  (entspricht dem `alignment_mode="expand"` des Related-Work-Skripts).

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

raw = load_dataset(DATASET, trust_remote_code=True)
print(raw)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)


def encode_example(example):
    text  = example["sentence"]
    ner   = example["ner_labels"]
    spans = list(zip(ner["start"], ner["stop"], ner["ner_class"]))

    enc = tokenizer(
        text,
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )

    labels = []
    prev_key = None  # (start, stop) der Entitaet des vorherigen Tokens
    for (tok_start, tok_end) in enc["offset_mapping"]:
        if tok_start == tok_end:            # Spezial-Token
            labels.append(-100)
            prev_key = None
            continue

        matched = None
        for (s, e, c) in spans:             # erstes ueberlappendes Entity gewinnt
            if tok_start < e and tok_end > s:
                matched = (s, e, c)
                break

        if matched is None:
            labels.append(LABEL2ID["O"])
            prev_key = None
        else:
            s, e, c = matched
            cls = ID2CLASS[int(c)]
            if prev_key == (s, e):
                labels.append(LABEL2ID[f"I-{cls}"])
            else:
                labels.append(LABEL2ID[f"B-{cls}"])
            prev_key = (s, e)

    enc.pop("offset_mapping")
    enc["labels"] = labels
    return enc


tokenized = raw.map(
    encode_example,
    remove_columns=raw["train"].column_names,
    desc="Tokenisieren + BIO-Alignment",
)
print(tokenized)

# Kurzer Blick auf ein Beispiel
ex = tokenized["train"][0]
toks = tokenizer.convert_ids_to_tokens(ex["input_ids"])
print("\nBeispiel-Alignment:")
for t, l in list(zip(toks, ex["labels"]))[:25]:
    print(f"  {t:<20} {ID2LABEL.get(l, 'IGN') if l != -100 else 'IGN'}")

## Modell: gbert-Encoder + Linear + **CRF**

Der Encoder liefert kontextualisierte Token-Repraesentationen, der Linear-Layer die
**Emissionen** (pro Token ein Score je Label). Der CRF modelliert zusaetzlich die
**Uebergangs-Scores** zwischen Labels und liefert die wahrscheinlichste *Gesamtsequenz*
statt tokenweise unabhaengiger Argmax-Entscheidungen.

- **Loss** = negative Log-Likelihood der Gold-Sequenz unter dem CRF.
- **Maske** = `attention_mask` (inkl. `[CLS]`/`[SEP]`, exkl. Padding). `pytorch-crf`
  verlangt, dass die erste Zeitposition unmaskiert ist — `[CLS]` an Position 0 erfuellt
  das. Maskierte/`-100`-Positionen werden fuer den Verlust auf `O` (ID 0) gesetzt, sind
  aber via Maske vom Score ausgeschlossen.

In [ ]:
import torch.nn as nn
from torchcrf import CRF
from transformers import AutoModel, AutoConfig
from transformers.modeling_outputs import TokenClassifierOutput


class GbertCrfForTokenClassification(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.1):
        super().__init__()
        self.num_labels = num_labels
        self.config  = AutoConfig.from_pretrained(model_name, num_labels=num_labels)
        self.encoder = AutoModel.from_pretrained(model_name, config=self.config)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, input_ids=None, attention_mask=None,
                token_type_ids=None, labels=None, **kwargs):
        enc_out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
        )
        sequence_output = self.dropout(enc_out.last_hidden_state)
        emissions = self.classifier(sequence_output)          # [B, T, C]
        mask = attention_mask.bool()

        loss = None
        if labels is not None:
            crf_labels = labels.clone()
            crf_labels[labels == -100] = 0                     # maskierte Pos. -> O
            loss = -self.crf(emissions, crf_labels, mask=mask, reduction="mean")

        return TokenClassifierOutput(loss=loss, logits=emissions)

    @torch.no_grad()
    def decode(self, input_ids=None, attention_mask=None, token_type_ids=None, **kwargs):
        # Beste Label-Sequenz je Beispiel (Viterbi) als Liste von Listen.
        enc_out = self.encoder(input_ids=input_ids, attention_mask=attention_mask,
                               token_type_ids=token_type_ids)
        emissions = self.classifier(enc_out.last_hidden_state)
        return self.crf.decode(emissions, mask=attention_mask.bool())


def build_model(dropout=0.1, model_name=None):
    return GbertCrfForTokenClassification(model_name or MODEL_NAME, NUM_LABELS, dropout)

## Metriken (seqeval) & CRF-Trainer

Zwei Anpassungen am HuggingFace-`Trainer`:

1. **`create_optimizer`** — getrennte Parametergruppen: Encoder mit der Basis-Lernrate,
   Kopf (**Linear + CRF**) mit einer eigenen, i. d. R. hoeheren Lernrate (`crf_lr`).
   Das ist Teil des HPO-Buendels.
2. **`prediction_step`** — dekodiert im Eval die Sequenzen via **Viterbi** (`crf.decode`)
   statt `argmax` der Emissionen, damit die Metriken die Uebergangs-Scores beruecksichtigen.

`label_names=["labels"]` ist zwingend, da das Modell kein `PreTrainedModel` ist — sonst
reicht der Trainer die Labels nicht an Loss/Metrik durch.

In [ ]:
import numpy as np
from torch.optim import AdamW
from transformers import Trainer
from transformers.trainer_pt_utils import get_parameter_names
from seqeval.metrics import precision_score, recall_score, f1_score, classification_report


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds  = np.asarray(preds)
    labels = np.asarray(labels)
    if preds.ndim == 3:                     # Fallback: rohe Emissionen -> argmax
        preds = preds.argmax(-1)

    true_labels, true_preds = [], []
    for p_row, l_row in zip(preds, labels):
        cur_p, cur_l = [], []
        for p, l in zip(p_row, l_row):
            if l == -100:
                continue
            cur_l.append(ID2LABEL[int(l)])
            cur_p.append(ID2LABEL[int(p)] if int(p) >= 0 else "O")
        true_labels.append(cur_l)
        true_preds.append(cur_p)

    return {
        "precision": precision_score(true_labels, true_preds),
        "recall":    recall_score(true_labels, true_preds),
        "f1":        f1_score(true_labels, true_preds),
    }


class CRFTrainer(Trainer):
    def __init__(self, *args, crf_lr=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.crf_lr = crf_lr if crf_lr is not None else self.args.learning_rate

    def create_optimizer(self):
        if self.optimizer is not None:
            return self.optimizer
        decay = set(n for n in get_parameter_names(self.model, [nn.LayerNorm])
                    if "bias" not in n)
        head = ("classifier", "crf")

        def is_head(n):
            return n.startswith(head)

        named = list(self.model.named_parameters())
        groups = [
            {"params": [p for n, p in named if n in decay and not is_head(n)],
             "weight_decay": self.args.weight_decay, "lr": self.args.learning_rate},
            {"params": [p for n, p in named if n not in decay and not is_head(n)],
             "weight_decay": 0.0, "lr": self.args.learning_rate},
            {"params": [p for n, p in named if n in decay and is_head(n)],
             "weight_decay": self.args.weight_decay, "lr": self.crf_lr},
            {"params": [p for n, p in named if n not in decay and is_head(n)],
             "weight_decay": 0.0, "lr": self.crf_lr},
        ]
        self.optimizer = AdamW(
            groups,
            betas=(self.args.adam_beta1, self.args.adam_beta2),
            eps=self.args.adam_epsilon,
        )
        return self.optimizer

    def prediction_step(self, model, inputs, prediction_loss_only, ignore_keys=None):
        inputs = self._prepare_inputs(inputs)
        core = model.module if hasattr(model, "module") else model
        with torch.no_grad():
            out = model(**inputs)
            loss = out.loss.detach() if out.loss is not None else None
            mask = inputs["attention_mask"].bool()
            decoded = core.crf.decode(out.logits, mask=mask)   # Viterbi

        if prediction_loss_only:
            return (loss, None, None)

        T = out.logits.shape[1]
        preds = torch.zeros((len(decoded), T), dtype=torch.long)  # 0 = O
        for i, seq in enumerate(decoded):
            preds[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)
        return (loss, preds, inputs.get("labels"))

## Data-Collator & TrainingArguments-Helfer

`DataCollatorForTokenClassification` paddet dynamisch und fuellt Label-Padding mit
`-100`. Der Helfer baut konsistente `TrainingArguments` fuer HPO- und Finaltraining.

In [ ]:
from transformers import DataCollatorForTokenClassification, TrainingArguments

collator = DataCollatorForTokenClassification(tokenizer=tokenizer, label_pad_token_id=-100)


def make_args(output_dir, lr, wd, warmup_ratio, epochs, batch_size, seed=SEED):
    return TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        weight_decay=wd,
        warmup_ratio=warmup_ratio,
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=max(batch_size, 16),
        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        label_names=["labels"],          # Modell ist kein PreTrainedModel!
        remove_unused_columns=False,
        report_to="none",
        fp16=FP16,
        seed=seed,
        dataloader_num_workers=2,
        disable_tqdm=True,
    )

## HPO-Buendel (Optuna)

**Suchraum**

| Hyperparameter | Bereich | Begruendung |
|---|---|---|
| `learning_rate` (Encoder) | 1e-5 – 7e-5 (log) | Transformer-Feintuning-Fenster |
| `crf_lr_mult` | 1 – 20 (log) | Kopf/CRF lernt i. d. R. schneller als der Encoder |
| `weight_decay` | 0.0 – 0.1 | Regularisierung |
| `warmup_ratio` | 0.0 – 0.2 | stabilerer Start |
| `dropout` | 0.1 – 0.4 | Regularisierung des Kopfes |
| `batch_size` | {8, 16} | GPU-Speicher vs. Gradientenrauschen |
| `num_train_epochs` | 2 – 4 | Under-/Overfitting |

TPE-Sampler steuert die Suche, MedianPruner bricht schwache Trials frueh ab. Zur
Colab-Tauglichkeit laeuft jeder Trial auf einem Train-Subset (`HPO_TRAIN_LIMIT`);
das finale Training nutzt danach den vollen Datensatz.

In [ ]:
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

hpo_train = tokenized["train"].shuffle(seed=SEED).select(
    range(min(HPO_TRAIN_LIMIT, len(tokenized["train"]))))
hpo_dev = tokenized["validation"]
print(f"HPO-Train: {len(hpo_train)}  |  HPO-Dev: {len(hpo_dev)}")


def objective(trial):
    lr           = trial.suggest_float("learning_rate", 1e-5, 7e-5, log=True)
    crf_lr_mult  = trial.suggest_float("crf_lr_mult", 1.0, 20.0, log=True)
    wd           = trial.suggest_float("weight_decay", 0.0, 0.1)
    warmup       = trial.suggest_float("warmup_ratio", 0.0, 0.2)
    dropout      = trial.suggest_float("dropout", 0.1, 0.4)
    batch_size   = trial.suggest_categorical("batch_size", [8, 16])
    epochs       = HPO_EPOCHS if TEST else trial.suggest_int("num_train_epochs", 2, 4)

    args = make_args(
        output_dir=f"/content/hpo/trial_{trial.number}",
        lr=lr, wd=wd, warmup_ratio=warmup, epochs=epochs, batch_size=batch_size,
    )
    trainer = CRFTrainer(
        model=build_model(dropout=dropout),
        args=args,
        train_dataset=hpo_train,
        eval_dataset=hpo_dev,
        data_collator=collator,
        compute_metrics=compute_metrics,
        crf_lr=lr * crf_lr_mult,
    )
    trainer.train()
    f1 = trainer.evaluate()["eval_f1"]

    del trainer
    torch.cuda.empty_cache()
    return f1


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_warmup_steps=1),
    study_name="gbert_crf_ner",
)
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

print("\n==== HPO fertig ====")
print("Bester Dev-F1:", round(study.best_value, 4))
print("Beste Parameter:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")

## Beste Parameter uebernehmen

Fallback-Werte, falls die HPO-Zelle uebersprungen wurde — so bleibt das Notebook auch
ohne Studie lauffaehig.

In [ ]:
try:
    best = dict(study.best_params)
except NameError:
    best = {}

BEST = {
    "learning_rate":    best.get("learning_rate", 3e-5),
    "crf_lr_mult":      best.get("crf_lr_mult", 5.0),
    "weight_decay":     best.get("weight_decay", 0.01),
    "warmup_ratio":     best.get("warmup_ratio", 0.1),
    "dropout":          best.get("dropout", 0.2),
    "batch_size":       best.get("batch_size", 16),
    "num_train_epochs": best.get("num_train_epochs", FINAL_EPOCHS),
}
print("Verwendete Hyperparameter fuers Finaltraining:")
BEST

## Finales Training auf dem vollen Datensatz + Test-Evaluierung

Training mit den besten Hyperparametern auf `train`, Evaluierung auf `test`. Bei mehreren
Seeds (`FINAL_SEEDS`) wird je Seed ein Modell trainiert und das beste (nach Test-F1)
behalten. Gespeichert werden State-Dict, Tokenizer, Label-Mapping und Metriken — sowohl
im Repo-`models`-Ordner als auch nach Google Drive.

In [ ]:
import json, shutil

def save_model(model, out_dir, extra=None):
    os.makedirs(out_dir, exist_ok=True)
    torch.save(model.state_dict(), os.path.join(out_dir, "pytorch_model.bin"))
    tokenizer.save_pretrained(out_dir)
    cfg = {
        "model_name": MODEL_NAME,
        "num_labels": NUM_LABELS,
        "label_list": LABEL_LIST,
        "label2id":   LABEL2ID,
        "id2label":   {str(k): v for k, v in ID2LABEL.items()},
        "max_length": MAX_LENGTH,
        "architecture": "GbertCrfForTokenClassification",
        "hyperparameters": BEST,
    }
    if extra:
        cfg.update(extra)
    with open(os.path.join(out_dir, "crf_config.json"), "w") as f:
        json.dump(cfg, f, indent=2, ensure_ascii=False)


REPO_MODELS = "/content/information_extraction_german_medical_data/related_work/models"
best_overall = {"f1": -1.0, "dir": None, "seed": None, "report": None}

for seed in FINAL_SEEDS:
    print(f"\n==================== FINAL seed {seed} ====================")
    args = make_args(
        output_dir=f"/content/final/seed_{seed}",
        lr=BEST["learning_rate"], wd=BEST["weight_decay"],
        warmup_ratio=BEST["warmup_ratio"], epochs=BEST["num_train_epochs"],
        batch_size=BEST["batch_size"], seed=seed,
    )
    model = build_model(dropout=BEST["dropout"])
    trainer = CRFTrainer(
        model=model, args=args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        data_collator=collator,
        compute_metrics=compute_metrics,
        crf_lr=BEST["learning_rate"] * BEST["crf_lr_mult"],
    )
    trainer.train()

    test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
    print("Test:", {k: round(v, 4) for k, v in test_metrics.items()
                    if k.startswith("test_") and isinstance(v, float)})

    name = f"gbert-large-crf-seed{seed}" + ("-test" if TEST else "")
    out_dir = os.path.join(REPO_MODELS, name)
    save_model(model, out_dir, extra={"test_metrics": test_metrics, "seed": seed})
    with open(os.path.join(out_dir, "test_metrics.json"), "w") as f:
        json.dump(test_metrics, f, indent=2)

    # nach Drive sichern (uebersteht Colab-Disconnect)
    zip_path = shutil.make_archive(os.path.join(DRIVE_DIR, name), "zip", out_dir)
    print("Gespeichert in Drive:", zip_path)

    if test_metrics["test_f1"] > best_overall["f1"]:
        best_overall = {"f1": test_metrics["test_f1"], "dir": out_dir, "seed": seed,
                        "report": None}

    del trainer, model
    torch.cuda.empty_cache()

print("\nBestes Modell:", best_overall["dir"], "| Test-F1:", round(best_overall["f1"], 4))

## Detaillierter Report & Beispiel-Inferenz

Klassenweiser seqeval-Report des besten Modells und eine Beispiel-Vorhersage.

In [ ]:
from seqeval.metrics import classification_report

# Bestes Modell laden
model = build_model(dropout=BEST["dropout"])
state = torch.load(os.path.join(best_overall["dir"], "pytorch_model.bin"),
                   map_location=DEVICE)
model.load_state_dict(state)
model.to(DEVICE).eval()

# Test-Report
eval_args = make_args("/content/report", BEST["learning_rate"], 0.0, 0.0, 1,
                      BEST["batch_size"])
rep_trainer = CRFTrainer(model=model, args=eval_args, data_collator=collator,
                         compute_metrics=compute_metrics)
pred = rep_trainer.predict(tokenized["test"])
y_pred, y_true = [], []
for p_row, l_row in zip(np.asarray(pred.predictions), np.asarray(pred.label_ids)):
    cp, cl = [], []
    for p, l in zip(p_row, l_row):
        if l == -100:
            continue
        cl.append(ID2LABEL[int(l)])
        cp.append(ID2LABEL[int(p)] if int(p) >= 0 else "O")
    y_pred.append(cp); y_true.append(cl)
print(classification_report(y_true, y_pred, digits=4))


def predict(text):
    enc = tokenizer(text, return_offsets_mapping=True, truncation=True,
                    max_length=MAX_LENGTH, return_tensors="pt")
    offsets = enc.pop("offset_mapping")[0].tolist()
    tags = model.decode(**{k: v.to(DEVICE) for k, v in enc.items()})[0]
    ents, cur = [], None
    for (start, end), tag_id in zip(offsets, tags):
        if start == end:
            continue
        tag = ID2LABEL[tag_id]
        if tag.startswith("B-"):
            if cur: ents.append(cur)
            cur = {"label": tag[2:], "start": start, "end": end}
        elif tag.startswith("I-") and cur and cur["label"] == tag[2:]:
            cur["end"] = end
        else:
            if cur: ents.append(cur); cur = None
    if cur: ents.append(cur)
    return [(text[e["start"]:e["end"]], e["label"]) for e in ents]


print(predict("Der Patient erhielt 500 mg Ibuprofen bei Kopfschmerzen und Bluthochdruck."))